# Hardware Architecture Documentation

## Introduction
#### Project GitHub Link: https://github.com/JDPennockGB/ENGR357-SDR-PennockRoque

This notebook serves as the primary hardware architecture documentation for a high-performance Software Defined Radio (SDR) receiver front-end, designed around a Tayloe Detector switching quad mixer and controlled by a Raspberry Pi Pico. The architecture is engineered to achieve clean quadrature ($I/Q$) baseband downconversion from radio frequency (RF) inputs, with specific optimization for low-noise performance, high dynamic range, and broad frequency coverage spanning from the standard AM broadcast band through the upper High Frequency (HF) amateur radio bands. By documenting the physical signal path, active filtering characteristics, and a unique 3-stage clock generation tree, this reference provides the mathematical and engineering foundations necessary to understand, debug, and interface with the physical 
hardware layer.

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/3D PCB Viewer.png" width="420" height="470" />
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/FullSchematic.png" width="700" height="470" />
</div>

---

## Major Project Design Hurdle: Switching Mixer to Tayloe Detector Pivot

A critical pivot occurred late in the hardware design phase regarding the receiver's downconversion topology. The original design architecture specified a standard **active switching mixer** integrated into a fully analog signal conditioning path, including branches into the ADC, as well as a speaker/headphone output path. However, **LTspice simulations** of the initial design revealed performance bottlenecks that forced a complete re-evaluation of the front-end architecture after the in-class design reviews, putting us behind the 8 ball for hardware design, preventing us from fully verifying our board before having to order from JLCPCB.

### Technical Failure Modes Identified in LTspice Simulations

When evaluating the traditional switching mixer layout under realistic RF signal conditions in LTspice, several severe degradations in signal integrity became apparent, which forced us to switch designs very late into the design process.

### The Engineering Solution: The Tayloe Detector

To overcome these simulation-proven bottlenecks, the architecture was completely redesigned around a **Tayloe Detector**, which we will talk about below. Due to how late we had to switch methodologies, we had 11 days between when we started our Tayloe detector design to when the PCB was finished and sent off to JLCPCB.

---

## Tayloe Detector Front-End

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/TayloeDetectorCircuit.png" width="1125" height="325" />
</div>

### 1. Circuit Overview & Signal Chain

The Tayloe detector (or switching quad mixer) transforms an RF signal into Quadrature (I/Q) baseband signals by sampling the input at 4 times the target carrier frequency ($4 \times f_{LO}$). 

#### RF to Baseband Signal Path
1. **Antenna Input & Protection:** ESD/Overvoltage protection via 1N4004W diodes.
2. **DC Blocking:** A $10\mu\text{F}$ centering capacitor removes any external DC offset.
3. **Impedance Matching & Isolation:** An **H1102NLT** transformer handles balanced-to-unbalanced conversion, with its secondary center tap biased at $V_{cc} / 2$ ($\approx 1.65\text{V}$).
4. **Quadrature Sampling (The Core):** An **SN74CBT3253DBQR** dual 4:1 multiplexer routes the RF signal sequentially to four holding capacitors, driven by $I\_CLK$ and $Q\_CLK$.
5. **Amplification & Filtering:** An **OPA2350** low-noise operational amplifier provides gain and low-pass filtering for the differential $I$ and $Q$ channels.
6. **Output Buffering & Bias:** A $1\mu\text{F}$ centering capacitor blocks DC, followed by a re-injection of the $V_{cc}/2$ bias to condition the $I\_OUT$ and $Q\_OUT$ signals for the Pi Pico's ADC range (0V - 3.3V).

### 2. Power Distribution & Bias Network Calculations

To achieve a high Dynamic Range and low Noise Floor, the analog front-end utilizes extensive decoupling and a stabilized active/passive bias network.

#### Bias Network Design
The midpoint bias ($V_{bias} = \frac{3.3\text{V}}{2} = 1.65\text{V}$) is generated via a $10\text{k}\Omega$ voltage divider. To maximize stability:
* Each power pin features a ferrite bead and a $10\mu\text{F}$ bypass capacitor to ground.
* A unique secondary $10\mu\text{F}$ capacitor is placed directly between the $3.3\text{V}$ rail and the $1.65\text{V}$ bias node to stabilize the reference against high-frequency switching transients.

Verification of the static power consumption of the divider and its source impedance:

In [21]:
import numpy as np

# System Parameters
V_cc = 3.3  # Volts
R1 = 10e3   # 10k Ohms
R2 = 10e3   # 10k Ohms

# 1. Calculate Bias Voltage
V_bias = V_cc * (R2 / (R1 + R2))

# 2. Calculate Static Current Draw through the divider
I_bias_static = V_cc / (R1 + R2)

# 3. Calculate Equivalent Output Impedance (Thevenin Resistance) of the divider
R_th = (R1 * R2) / (R1 + R2)

print(f"--- Bias Network Analysis ---")
print(f"Nominal Bias Voltage:  {V_bias:.2f} V")
print(f"Static Current Draw:   {I_bias_static * 1e3:.3f} mA")
print(f"Source Impedance:      {R_th / 1e3:.2f} kΩ")

--- Bias Network Analysis ---
Nominal Bias Voltage:  1.65 V
Static Current Draw:   0.165 mA
Source Impedance:      5.00 kΩ


### 3. Amplification & Low-Pass Filter (LPF) Analysis

The **OPA2350** operational amplifier is configured as a two-stage conditioning circuit for each channel ($I$ and $Q$) to isolate, amplify, and filter the baseband signals before ADC sampling:

* **Stage 1 (Unity Gain Buffer):** Configured as a voltage follower with the input routed to the non-inverting (+) terminal via a $10\,\Omega$ isolation resistor, and the output tied directly back to the inverting (-) terminal. This provides a very high input impedance to prevent loading the Tayloe mixer, and a low output impedance to drive the next stage.
* **Stage 2 (Active Low-Pass Filter & Gain Stage):** Configured as an inverting amplifier. The signal from Stage 1 enters the inverting terminal via a $47\,\Omega$ resistor ($R_{in}$). The feedback network consists of a $1.5\,\text{k}\Omega$ resistor ($R_{fb}$) in parallel with a $47\,\text{pF}$ capacitor ($C_{fb}$) to provide both voltage gain and anti-aliasing low-pass filtering. The non-inverting terminal is tied to ground/bias through a $10\,\Omega$ resistor.

Let's use Python to calculate the exact mid-band voltage gain and the $-3\,\text{dB}$ cutoff frequency of this specific topology.

In [22]:
R_in = 47.0       # Stage 2 Input Resistance (Ohms)
R_fb = 1500.0     # Stage 2 Feedback Resistance (Ohms)
C_fb = 47e-12     # Stage 2 Feedback Capacitance (47 pF)

# Stage 1 Gain (Voltage Follower)
A_v1 = 1.0

# Stage 2 Gain (Inverting Amplifier: |Av| = R_fb / R_in)
A_v2 = R_fb / R_in

# Total Cascaded System Gain
A_v_total = A_v1 * A_v2
A_v_db = 20 * np.log10(A_v_total)

# Filter Cutoff Frequency (-3dB point determined by Stage 2 feedback network)
f_cutoff = 1.0 / (2.0 * np.pi * R_fb * C_fb)

print(f"--- OPA2350 Two-Stage Analysis ---")
print(f"Stage 1 Voltage Gain:  {A_v1:.1f} (Buffer)")
print(f"Stage 2 Voltage Gain:  {A_v2:.2f}")
print(f"Total Voltage Gain:    {A_v_total:.2f} Linear ({A_v_db:.2f} dB)")
print(f"Filter Cutoff (-3dB):  {f_cutoff / 1e6:.2f} MHz ({f_cutoff / 1e3:.2f} kHz)")

--- OPA2350 Two-Stage Analysis ---
Stage 1 Voltage Gain:  1.0 (Buffer)
Stage 2 Voltage Gain:  31.91
Total Voltage Gain:    31.91 Linear (30.08 dB)
Filter Cutoff (-3dB):  2.26 MHz (2257.52 kHz)


---
## Clock Generation & Quadrature Architecture

The switching execution of the **SN74CBT3253DBQR** Tayloe Mux relies on two quadrature clocks ($I\_CLK$ and $Q\_CLK$) that must be exactly $90^\circ$ out of phase. To achieve stable tuning across high-frequency (HF) amateur bands down to the lower AM broadcast frequencies, a 3-stage clock synthesis architecture is implemented.

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/CMOSsi5351Circuit.png" width="650" height="312" />
</div>

#### Clock Tree Stages
1. **Primary Reference (Stage 1):** A **YC24.576MUBCE2O** CMOS crystal oscillator provides an ultra-stable $24.576\,\text{MHz}$ master local oscillator clock ($LO\_CLK$). This signal passes through a $33\,\Omega$ source-termination resistor to minimize transmission line reflections before driving the clock generator.
2. **Frequency Synthesis and Quadrature Generation (Stage 2):** An **Si5351A** configurable clock generator accepts the $LO\_CLK$ reference. It is controlled by the Pi Pico over an $\text{I}^2\text{C}$ bus (`OSC_SCL` / `OSC_SDA`). While the Si5351A features a native quadrature phase-shift mode, its internal phase-shifters limit its minimum output frequency to $3.2\,\text{MHz}$ in this mode.
3. **Frequency Division (Stage 3):** To bypass the $3.2\,\text{MHz}$ limitation and access the lower AM and shortwave bands, an **SN74AC74DR** dual D-flip-flop is configured as a hardware **Johnson Counter**. By running the Si5351A output at $4\times$ the required Tayloe switching frequency, the Johnson counter cleanly divides the clock rate by 4. This step guarantees perfect $50\%$ duty cycles and precise $90^\circ$ phase spacing. The final outputs pass through $33\,\Omega$ series-damping resistors to eliminate high-frequency ringing before entering the Tayloe Mux control pins as $I\_CLK$ and $Q\_CLK$.

Let's use Python to compute the relationship between the Si5351A output frequency, the resulting Tayloe sampling rate, and the final RF center frequency ($f_{RF}$).

In [23]:
f_si5351_min = 3.2e6      # 3.2 MHz hardware limit
f_si5351_max = 160e6      # Stable upper limit for Si5351 high-frequency output

johnson_divider = 4

def calculate_rf_coverage(f_clock):
    """
    Calculates the resulting Tayloe switching clock and the target RF tuning frequency.
    For a Tayloe detector, the multiplexer switches at 4x the target RF carrier frequency.
    Therefore: f_RF = f_switching / 4
    With the Johnson counter in place: f_switching = f_clock / 4
    Thus: f_RF = f_clock / 16
    """
    f_switching = f_clock / johnson_divider
    f_rf = f_switching / 4
    return f_switching, f_rf

# Calculate operational limits
f_switch_min, f_rf_min = calculate_rf_coverage(f_si5351_min)
f_switch_max, f_rf_max = calculate_rf_coverage(f_si5351_max)

print(f"--- Clock Architecture & RF Coverage Analysis ---")
print(f"Si5351A Output Range:     {f_si5351_min / 1e6:.3f} MHz to {f_si5351_max / 1e6:.1f} MHz")
print(f"Tayloe Switching Range:   {f_switch_min / 1e3:.2f} kHz to {f_switch_max / 1e6:.2f} MHz")

--- Clock Architecture & RF Coverage Analysis ---
Si5351A Output Range:     3.200 MHz to 160.0 MHz
Tayloe Switching Range:   800.00 kHz to 40.00 MHz


#### Johnson Counter Output Clock Lines & Multiplexer Routing

The table below illustrates how the **SN74AC74DR** outputs progress through their sequence, showing how the divided high-speed clock yields perfectly synchronized quadrature phases to route the RF input across the **SN74CBT3253DBQR** multiplexer channels:

| Master Clock Cycle | Flip-Flop Q1 ($I\_CLK$) | Flip-Flop Q2 ($Q\_CLK$) | Resulting Phase Angle | Targeted Mux State / Signal Routed |
| :---: | :---: | :---: | :---: | :--- |
| **0** | 0 | 0 | $0^\circ$ | **00**: $I^+$ ($0^\circ$ sample) |
| **1** | 1 | 0 | $90^\circ$ | **01**: $Q^+$ ($90^\circ$ sample) |
| **2** | 1 | 1 | $180^\circ$ | **10**: $I^-$ ($180^\circ$ sample) |
| **3** | 0 | 1 | $270^\circ$ | **11**: $Q^-$ ($270^\circ$ sample) |

In **Notebook 2 (Software Design Architecture)**, we will detail how the Pi Pico's firmware controls the Si5351A over $\text{I}^2\text{C}$, schedules the ADC DMA transfers, and processes the incoming $I\_OUT$ and $Q\_OUT$ baseband channels from the operational amplifiers.

---

## I and Q Digitization: PCM1808 ADC

To process the downconverted analog $I$ and $Q$ baseband signals digitally, the hardware routes `I_OUT` and `Q_OUT` from the operational amplifier stages into a **PCM1808** high-performance, 24-bit stereo Analog-to-Digital Converter (ADC). The PCM1808 digitizes the channels simultaneously, preserving the vital phase relationship required for quadrature demodulation.

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/ADCcircuit.png" width="467" height="350" />
</div>

#### Anti-Aliasing Input Filter Topology
High-speed delta-sigma ADCs are susceptible to high-frequency noise folding back into the audible baseband passband (aliasing). To prevent this, a passive, low-pass $RC$ anti-aliasing filter is placed directly at the input pins (`VINL` and `VINR`) of the PCM1808 for both channels:
* **Series Resistance ($R$):** A $470\,\Omega$ isolation resistor is placed in series with the signal line.
* **Shunt Capacitance ($C$):** A $2.2\,\text{nF}$ capacitor is placed from the ADC input pin side directly to Ground ($0\,\text{V}$).

This passive network serves two critical purposes: it provides a secondary pole of high-frequency filtering after the active OPA2350 stage, and it acts as a charge-reservoir buffer to decouple the op-amp outputs from the switched-capacitor input sampling circuitry internal to the PCM1808.

Let's use Python to verify the exact cutoff frequency of this input filter network.

In [24]:
R_adc = 470       # Series Resistor (470 Ohms)
C_adc = 2.2e-9      # Shunt Capacitor (2.2 nF)

# Calculate the passive -3dB cutoff frequency
f_cutoff_adc = 1.0 / (2.0 * np.pi * R_adc * C_adc)

print(f"--- PCM1808 Input Anti-Aliasing Filter Analysis ---")
print(f"Series Resistance:      {R_adc:.1f} Ω")
print(f"Shunt Capacitance:     {C_adc * 1e9:.1f} nF")
print(f"Passive Cutoff: {f_cutoff_adc / 1e3:.2f} kHz ({f_cutoff_adc / 1e6:.4f} MHz)")

--- PCM1808 Input Anti-Aliasing Filter Analysis ---
Series Resistance:      470.0 Ω
Shunt Capacitance:     2.2 nF
Passive Cutoff: 153.92 kHz (0.1539 MHz)


---

## User Interface

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/EncoderButtonsCircuit.png" width="492" height="350" />
</div>

### 1. EC11E18244AU Rotary Encoder

To facilitate real-time user interaction, such as frequency tuning, filter bandwidth adjustment, or menu navigation, the hardware incorporates an **ECllE18244AU** rotary encoder featuring a built-in momentary push button. A mechanical encoder with an integrated switch was selected specifically for design versatility, minimizing the required physical footprint while maximizing user control options.

#### Interface Configuration & Pull-Up Network
Mechanical encoders and switches require hardware pull-up or pull-down networks to establish deterministic logic levels when the internal contacts are open. For this design:
* **`Enc_A` (Channel A):** Connected to a $10\,\text{k}\Omega$ pull-up resistor tied to the $3.3\,\text{V}$ rail.
* **`Enc_B` (Channel B):** Connected to a $10\,\text{k}\Omega$ pull-up resistor tied to the $3.3\,\text{V}$ rail.
* **`Enc_Sw` (Push Button Switch):** Connected to a $10\,\text{k}\Omega$ pull-up resistor tied to the $3.3\,\text{V}$ rail.

When the encoder rotates or the button is pressed, the internal mechanical contacts pull the respective lines directly to Ground ($0\,\text{V}$), creating a distinct falling edge. When the contacts open, the $10\,\text{k}\Omega$ resistors pull the lines back up to a crisp logic high ($3.3\,\text{V}$).

#### Quadrature Decoding Principles
The rotary encoder outputs two digital signals, `Enc_A` and `Enc_B`, which are phase-shifted by $90^\circ$ relative to each other (Quadrature encoding). By monitoring which channel leads or lags, the microcontroller determines the direction of rotation.

| Direction of Rotation | First Falling Edge | Second Falling Edge | Phase Relationship |
| :--- | :---: | :---: | :--- |
| **Clockwise (CW)** | `Enc_A` | `Enc_B` | Channel A leads Channel B |
| **Counter-Clockwise (CCW)** | `Enc_B` | `Enc_A` | Channel B leads Channel A |

### 2. TS-1185-W-4P-H1.8-160 Push Button Array

In addition to the rotary encoder, the hardware features an auxiliary array of four **TS-1185-W-4P-H1.8-160** momentary tactile switches. These surface-mount switches are included to provide optional user interface (UI) navigation and control redundancy if needed—such as band selection, mode toggling (e.g., AM, USB, LSB), or stepping through memory channels.

#### Interface Configuration & Pull-Up Network
Similar to the rotary encoder, each tactile switch is configured with an active-low topology utilizing an external passive network to ensure clean digital state transitions:
* **Switch 1 through Switch 4:** Each independent digital line is tied to the $3.3\,\text{V}$ rail via its own $10\,\text{k}\Omega$ pull-up resistor.
* **Operation:** In their default idle state, the switches remain open, and the $10\,\text{k}\Omega$ resistors pull the respective Pi Pico GPIO inputs to a stable logic HIGH ($3.3\,\text{V}$). Upon physical depression, the internal dome contact shorts the line directly to Ground ($0\,\text{V}$), bringing the microcontroller input logic LOW.

#### GPIO Mapping and Interface Constraints

| Switch ID | Physical Part Number | Default State | Pressed State | Typical Assigned Function |
| :---: | :--- | :---: | :---: | :--- |
| **SW1** | TS-1185-W-4P-H1.8-160 | HIGH ($3.3\,\text{V}$) | LOW ($0\,\text{V}$) | Optional / Back |
| **SW2** | TS-1185-W-4P-H1.8-160 | HIGH ($3.3\,\text{V}$) | LOW ($0\,\text{V}$) | Optional / Scroll Left |
| **SW3** | TS-1185-W-4P-H1.8-160 | HIGH ($3.3\,\text{V}$) | LOW ($0\,\text{V}$) | Optional / Select |
| **SW4** | TS-1185-W-4P-H1.8-160 | HIGH ($3.3\,\text{V}$) | LOW ($0\,\text{V}$) | Optional / Scroll Right |

#### Software Considerations for Mechanical Switches
Because tactile switches rely on a physical metallic spring-dome mechanism, they are subject to intermittent contact chatter (mechanical switch bounce) lasting several milliseconds upon closure and release.

### 3. NHD-0108BZ-FSY-YBW-33V3 1x8 Character LCD

To provide real-time visual telemetry—such as the current tuned RF frequency, signal strength indication (S-meter), and active operating modes—the hardware integrates a **NHD-0108BZ-FSY-YBW-33V3** 1x8 character LCD. This specific model is designed for $3.3\,\text{V}$ operation, matching the native IO voltage of the Pi Pico without requiring logic-level translation.

#### Hardware Interface Constraints & PCB Revision Errata
During the routing of the current **v3 PCB revision**, an error occurred where the odd and even columns of the LCD footprint headers were accidentally swapped. 
* **Remediation:** To interface the physical display with the v3 PCB, an **external wiring adapter/breakout** must be used to un-swap the staggered lines and route them to their correct destination pins. 
* Full detailed mapping can be found in the project's root `README.md` file.

#### Contrast Configuration
Standard character LCDs utilize a contrast pin ($V_0$, Pin 3) to adjust fluid bias, usually driven via a variable potentiometer. In this design, to simplify the BOM and maximize display readability, the **contrast pin is tied directly to Ground ($0\,\text{V}$)**. This sets the liquid crystal drive to its maximum possible contrast configuration, ensuring readability across diverse lighting conditions.


### 4. Auxiliary Display Option - OLED I²C Header

For expanded display capability or future modular upgrades, the PCB includes an unpopulated 4-pin header specifically provisioned for a standard small-form-factor OLED display module (such as a $0.96\text{"}$ or $1.3\text{"}$ $128\times64$ SSD1306/SH1106 display). While this screen is not populated on the standard build, the routing is fully integrated into the hardware tracking for maximum prototyping flexibility.

#### Interface Configuration & Protection Network
The OLED expansion port utilizes the I2C bus. To preserve signal integrity across the bus and protect the Pi Pico’s native GPIO pins from fast transient spikes during a hot-plug event, series-damping resistors are included on the data lines:
* **Pin 1 ($V_{SS}$):** Tied directly to System Ground ($0\,\text{V}$).
* **Pin 2 ($V_{DD}$):** Tied directly to the $3.3\,\text{V}$ power bus.
* **Pin 3 (SCL):** Connected to the shared hardware $\text{I}^2\text{C}$ Clock line through a $100\,\Omega$ series isolation resistor.
* **Pin 4 (SDA):** Connected to the shared hardware $\text{I2C}$ Data line through a $100\,\Omega$ series isolation resistor.


<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/LCDOLEDcircuit.png" width="315" height="400" />
</div>

---

## I²C Bus

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/I2CbusCircuit.png" width="275" height="250" />
</div>

---

## Pi Pico Layout

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/PicoCircuit.png" width="475" height="480" />
</div>

---

## Voltage Measurements and Verification

### Voltage Busses
- 3.3v @ Pico Output - 3.300v +-0.0005v
- 3.3v @ Bus Probe Point - 3.300v
- 3.3v @ Header Pin - 3.300v
- 5v @ Pico Output - 5.249v 
- 5v @ Bus Probe Point - 5.2485v  +-0.0005v
- 5v @ Header Pin - 5.249v

### PCM1808 - FULLY STABLE
- VREF, Pin 1 - 3.300v
- VCC, Pin 3 - 3.300v
- VDD, Pin 4 - 3.300v
- MD0, Pin 10 - 3.300v

### OLED Header - FULLY STABLE
- VCC, Pin 2 - 3.300v

### LCD Header
- VCC, Pin 2 - 3.3005v +-0.0005v

### CMOS Oscillator - FULLY STABLE
- EN, Pin 1 - 3.299v
- VDD, Pin 4 - 3.299v

### si5351 I2C Oscillator
- VDD, Pin 1 - 3.2985v +-0.0005v
- VDDO, Pin 7 - 3.2995 +-0.0005v

### SN74AC Johnson Counter - FULLY STABLE
- 1CLR, Pin 1 - 3.300v
- 1PRE, Pin 4 - 3.300v
- 1PRE, Pin 10 - 3.300v
- 1CLR, Pin 13 - 3.300v
- VCC, Pin 14 - 3.300v

### H1102NLT Transformer - FULLY STABLE
- V_center, Pin 15 - 1.6472v

### SN74CB Mux
- VCC, Pin 16 - 5.2485v +-0.0005v

### OPA2350 Op-Amps - FULLY STABLE
- U4, V+, Pin 8 - 3.299v
- U5, V+, Pin 8 - 3.299v

### I/Q Recentering Bias Voltage - FULLY STABLE
- I_V_center, Pin 16 - 1.6525v
- Q_V_center, Pin 16 - 1.6483v

### Rotary Encoder w/ Button - FULLY STABLE
- Enc_B Pin A - 3.298v
- Enc_A, Pin B - 3.298v
- Enc_Sw, Pin E - 2.7333v

### Buttons - FULLY STABLE
- SW1, VCC, Pin 1 - 2.7390v
- SW2, VCC, Pin 1 - 2.7435v
- SW3, VCC, Pin 1 - 2.7396v
- SW4, VCC, Pin 1 - 2.7411v

---

## Clock Measurements & Verification

### 24.576 MHz CMOS Oscilator
- Frequency: 24.5762 MHz
- Vpp: 2.90v
- Vrms: 1.67V
- Waveform: Augmented Triangle Wave

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/CmosClock.JPG" width="400" height="270" />
</div>

### si5351 I2C Clock Output
- Frequency: 5.6855 MHz
- Vpp: 3.91v
- Vrms: 3.91v
- Waveform: Square wave with some jitter noise

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/si5351Clock.JPG" width="400" height="270" />
</div>

### I/Q Quadrature Clocks from Johnson Counter
- Frequency: 12.005 MHz
- Vpp: 3.87v
- Vrms: 3.87v
- Waveform: 2 square waves in quadrature

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/JohnsonClocks.JPG" width="400" height="270" />
</div>

### 12.000 MHz Pico Oscilator
- Frequency: 12.005 MHz
- Vpp: 1.31v
- Vrms: 450 mV
- Waveform: Sine wave

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/PicoClock.JPG" width="400" height="270" />
</div>

---

## SDR Firmware Architecture & Technical Approach

The firmware for this Software Defined Radio (SDR) receiver is built upon the Raspberry Pi Pico SDK. It utilizes a dual-core, non-blocking architecture to ensure uninterrupted high-fidelity audio streaming alongside responsive standalone user controls and USB serial communications.

### I2C Oscillator Control (si5351 Driver)
The radio's Variable Frequency Oscillator (VFO) is controlled dynamically via software. This builds upon the SDK’s `hardware_i2c` library. The firmware consists of a custom C driver (`si5351.c`) that calculates the Phase-Locked Loop (PLL) and Multisynth registers (N, P1, P2, P3) based on the target frequency. A critical software configuration is the strict use of timeout-based I2C calls (e.g., `i2c_write_timeout_us`). This prevents the familiar "I2C freeze"—ensuring that if the hardware bus momentarily hangs, the software gracefully times out rather than locking the main core and crashing the USB audio stream.

### Standalone UI: Rotary Encoder
To enable standalone frequency tuning without relying exclusively on the USB host, a software-based quadrature decoding system is integrated via the `hardware_gpio` library. The firmware utilizes an Interrupt Service Routine (ISR) configured to trigger on the edge states of the encoder's A and B pins. This ISR decodes rotational direction, safely increments or decrements the global `current_hz` target variable, and immediately invokes the I2C driver to update the si5351 oscillator in real-time.

### Standalone UI: 1x8 Alphanumerical LCD Display
To provide localized visual feedback of the radio’s operating state, a dedicated display rendering task runs in the primary execution loop. Because the SDR hardware utilizes a 4:1 Johnson counter to generate the quadrature I/Q signals, the software tracks the physical oscillator frequency at 4x the reception frequency. The non-blocking UI task polls the `current_hz` variable, mathematically divides it by 4 to reflect the true tuned frequency, formats it into a string (e.g., `1420 kHz`), and sends it to the display.

### I2S Audio Capture via PIO (PCM1808 Interface)
To digitize the analog baseband I/Q signals coming from the radio hardware, the software captures a high-speed I2S data stream from the PCM1808 ADC. This implementation builds upon the RP2040’s unique Programmable I/O (PIO) subsystem. The firmware utilizes a custom PIO assembly program (`i2s_rx.pio`) loaded into a state machine. It is configured to precisely synchronize with the ADC’s external Bit Clock (BCK) and Left/Right Word Clock (LRCK), shifting the incoming audio data into memory slots for efficient processor handling.

### UAC1 Audio Streaming, CDC Control (TinyUSB), and DMA Ping-Pong Buffering 
- To deliver the digitized radio spectrum to the host computer's SDR software (e.g., Quisk), the Pico acts as a composite USB device. It enumerates as a standard USB Audio Class 1 (UAC1) microphone alongside a virtual COM port. This builds upon the TinyUSB device stack. It works in many programs, namely on MacOS, which has more strict USB device standards that took quite a bit of debugging to get working on the board.

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/UAC1Enum.png" width="450" height="270" />
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/AudacityEnum.png" width="650" height="270" />
</div>

- **Audio Streaming:** This aspect of the project does not currently work. I am pretty sure there is some firmware issue between the DMA and the ADC, but just ran out of time debugging it. 

---

## Finished and (mostly) Working SDR Board

<div style="text-align: left;">
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/SDRinit.jpg" width="320" height="240" />
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/SDRmin.JPG" width="320" height="240" />
<img src="/Users/josephpennock/Documents/GitHub/ENGR-356-SDR/Images/SDRmax.JPG" width="320" height="240" />
</div>

---

## Future Improvements
- Correct the swapped columns of the LCD header so that the weird adapter I made doesn't need to be used anymore.
- 3D-Print a housing for the PCB and using the mounting holes to secure it.
- Further fix the ADC-DMA pipeline issue that is blocking the board from transmitting audio over USB.
- Remove the OLED header from the board since the LCD works better for the specific case.
- Add an analog in jack like on the 2026 board from CPTR 480 so that the audio can bre streamed in from a phone for more debugging options.
- Add a full analog listening and output system to make it so that you can use it independenlty.
- Use a dual Johnson counter so that we can get down to 400 kHz since we are clipping at 790 kHz due the the si5351 limitations.
- Add 33 Ohm resistor to LO_CLK to PCM1808 line, is currently missing